# 03 · Integração espacial e locais candidatos

**Objetivo.** Integrar demanda censitária e oferta escolar existentes, produzir candidatos teóricos para abertura e uma matriz de deslocamento consistente. O problema estudado é **expansão de rede existente**, não substituição da rede.

Os locais candidatos são **pontos internos dos setores censitários**; servem somente como alternativas geométricas em um experimento de otimização. Um setor NÃO significa disponibilidade real de terreno, aprovação para construção ou propriedade pública. A validação de terrenos e infraestrutura demandaria fontes municipais específicas.

Mantemos candidatos para **todos os setores com alguma demanda positiva**, sem amostragem, até que existam critérios de elegibilidade fundiária verificáveis. Isto pode elevar o custo computacional e gera alternativas potencialmente irreais; a interpretação deve reconhecer isso.

## 1. Importar saídas auditadas dos notebooks 01 e 02

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

raiz = Path.cwd().resolve()
if not (raiz / "pyproject.toml").exists():
    raiz = next((p for p in raiz.parents if (p / "pyproject.toml").exists()), raiz)
assert (raiz / "pyproject.toml").exists(), "Execute dentro do repositório PM."
saida = raiz / "outputs" / "localizacao_escolar" / "lavras"
saida.mkdir(parents=True, exist_ok=True)
CODIGO_MUNICIPIO = "3138203"
ETAPAS = {"fundamental_iniciais": ("V01032", "QT_MAT_FUND_AI", "QT_TUR_FUND_AI"),
          "fundamental_finais": ("V01033", "QT_MAT_FUND_AF", "QT_TUR_FUND_AF"),
          "medio": ("V01034", "QT_MAT_MED_PROP", "QT_TUR_MED_PROP")}
import geopandas as gpd
setores = pd.read_csv(saida / "setores_demanda.csv", dtype={"CD_SETOR": "string"})
escolas = pd.read_csv(saida / "escolas_auditadas.csv", dtype={"CO_ENTIDADE": "string"})
assert setores.CD_SETOR.is_unique and escolas.CO_ENTIDADE.is_unique
info_ibge = json.loads((saida / "metadados_ibge.json").read_text())
info_inep = json.loads((saida / "metadados_inep.json").read_text())
CRS_UTM = info_ibge["crs_projetado"]
assert (setores[["x_m", "y_m"]].notna().all()).all()
print("Setores:", len(setores), "| escolas cadastradas:", len(escolas), "| CRS:", CRS_UTM)

## 2. Verificar pontos escolares e projetá-los

Um endereço sem coordenada confiável não deve desaparecer silenciosamente. Usamos somente escolas elegíveis da etapa e exigimos coordenadas completas, com diagnóstico explícito. Uma revisão cadastral ainda é necessária antes de decisões reais.

In [ ]:
coords = gpd.GeoDataFrame(escolas.copy(),
                           geometry=gpd.points_from_xy(escolas.longitude, escolas.latitude),
                           crs="EPSG:4326").to_crs(CRS_UTM)
escolas["x_m"] = coords.geometry.x.to_numpy()
escolas["y_m"] = coords.geometry.y.to_numpy()
for etapa in ETAPAS:
    elegiveis = escolas.loc[escolas["elegivel_" + etapa].fillna(False)]
    problemas = elegiveis.loc[~elegiveis["coord_valida"].fillna(False),
                             ["CO_ENTIDADE", "NO_ENTIDADE", "fonte_coordenada"]]
    print(etapa, "| coordenadas pendentes:", len(problemas))
    if len(problemas):
        display(problemas)
# Ponto de partida: proximidade ao município, não validação de geocodificação fina.
limites = gpd.read_file(raiz / "data" / "BR_setores_CD2022.gpkg",
                       layer="BR_setores_CD2022",
                       where=f"CD_MUN = '{CODIGO_MUNICIPIO}'",
                       columns=["CD_SETOR"]).to_crs(CRS_UTM).geometry.union_all()
escolas["distancia_limite_m"] = coords.geometry.distance(limites).to_numpy()
escolas["dentro_municipio"] = coords.geometry.map(limites.covers).to_numpy()
for etapa in ETAPAS:
    sel = escolas.loc[escolas["elegivel_" + etapa].fillna(False)]
    suspeitos = sel.loc[sel.distancia_limite_m.gt(250), ["CO_ENTIDADE", "NO_ENTIDADE", "distancia_limite_m"]]
    if len(suspeitos):
        print("ATENÇÃO coordenadas >250 m fora do município", etapa)
        display(suspeitos)

## 3. Seleção dos candidatos (cenário exploratório)

Um candidato em cada setor com demanda positiva no cenário mais alto evita restringir artificialmente o espaço de busca. **Não representa lote ou endereço apto para uma escola.** A capacidade hipotética de uma nova unidade é \(U=T^{novo}\cdot s\), onde \(T^{novo}\) é um número ilustrativo de turmas e \(s\) é a hipótese de lotação utilizada na mesma etapa da rede existente. Escolhas de tamanho devem ser avaliadas por sensibilidade, não interpretadas como padrão escolar oficial.

In [ ]:
TURMAS_NOVA_ESCOLA = 12  # Hipótese exploratória, não projeto arquitetônico.
LOTACAO = info_inep["lotacao_por_turma"]
cand_mask = setores[[f"demanda_{etapa}_alta" for etapa in ETAPAS]].sum(axis=1).gt(0)
candidatos = setores.loc[cand_mask, ["CD_SETOR", "x_m", "y_m"]].copy()
candidatos["id"] = "N_" + candidatos.CD_SETOR
candidatos["origem"] = "ponto_representativo_setor_censitario"
assert candidatos.id.is_unique
print("Candidatos abstratos:", len(candidatos),
      "| Capacidade por escola nova (cenário central):", TURMAS_NOVA_ESCOLA * LOTACAO["central"])
display(candidatos.head())

## 4. Preparar instâncias por etapa

Para fins de modelagem, removemos setores de demanda **zero em todos os cenários** (eles continuam no inventário do notebook 01). As matrizes de distância usam UTM métrico: são distâncias em linha reta, não tempos ou trajetos rodoviários. O modelo diferencia `E_` (unidade existente) e `N_` (candidato).

In [ ]:
def matriz_distancias(origens, destinos):
    dx = origens.x_m.to_numpy(dtype=float)[:, None] - destinos.x_m.to_numpy(dtype=float)[None, :]
    dy = origens.y_m.to_numpy(dtype=float)[:, None] - destinos.y_m.to_numpy(dtype=float)[None, :]
    return pd.DataFrame(np.hypot(dx, dy) / 1000,
                        index=origens.id.to_list(), columns=destinos.id.to_list())

tabela_resumo = []
for etapa, (_, mat, tur) in ETAPAS.items():
    pasta = saida / "instancias" / etapa
    pasta.mkdir(parents=True, exist_ok=True)
    campos_demanda = [f"demanda_{etapa}_{x}" for x in ("baixa", "central", "alta")]
    dem = setores.loc[setores[campos_demanda].sum(axis=1).gt(0),
                      ["CD_SETOR", "x_m", "y_m", "SITUACAO", *campos_demanda]].copy()
    dem = dem.rename(columns={col: col.replace(f"demanda_{etapa}_", "w_") for col in campos_demanda})
    dem["id"] = dem.CD_SETOR.astype(str)
    of = escolas.loc[escolas["elegivel_" + etapa].fillna(False)].copy()
    assert len(of), f"Sem escolas elegíveis: {etapa}"
    assert of["coord_valida"].fillna(False).all(), f"Coordenadas INEP faltantes em {etapa}. Corrija auditando antes de modelar."
    assert of.distancia_limite_m.le(250).all(), f"Escolas fora do município >250 m em {etapa}: auditar."
    of["id"] = "E_" + of.CO_ENTIDADE.astype(str)
    of["matriculas_observadas"] = of[mat]
    of["turmas_observadas"] = of[tur]
    for nome, tamanho in LOTACAO.items():
        of["q_" + nome] = (of[tur] * tamanho).astype(int)
    of = of[["id", "CO_ENTIDADE", "NO_ENTIDADE", "x_m", "y_m",
             "matriculas_observadas", "turmas_observadas",
             *["q_" + x for x in LOTACAO], "fonte_coordenada", "precisao_coordenada"]]
    novos = candidatos.copy()
    for nome, tamanho in LOTACAO.items():
        novos["u_" + nome] = TURMAS_NOVA_ESCOLA * tamanho
    D = matriz_distancias(dem, pd.concat([of[["id", "x_m", "y_m"]],
                                          novos[["id", "x_m", "y_m"]]], ignore_index=True))
    assert np.isfinite(D.to_numpy()).all()
    dem.to_csv(pasta / "demanda.csv", index=False)
    of.to_csv(pasta / "existentes.csv", index=False)
    novos.to_csv(pasta / "candidatos.csv", index=False)
    D.rename_axis("id").to_csv(pasta / "distancias_km.csv")
    tabela_resumo.append({"etapa": etapa, "setores": len(dem),
                          "existentes": len(of), "candidatos": len(novos),
                          "demanda_central": int(dem.w_central.sum()),
                          "capacidade_existente_central": int(of.q_central.sum()),
                          "capacidade_nova_central": TURMAS_NOVA_ESCOLA * LOTACAO["central"],
                          "pares": int(D.size)})
tabela_resumo = pd.DataFrame(tabela_resumo)
display(tabela_resumo)
tabela_resumo.to_csv(saida / "resumo_instancias.csv", index=False)
(saida / "metadados_instancias.json").write_text(json.dumps({
    "nova_escola_turmas_hipoteticas": TURMAS_NOVA_ESCOLA,
    "candidatos": "pontos representativos em setores de demanda potencial positiva; terrenos nao verificados",
    "crs_projetado": CRS_UTM, "distancia": "euclidiana linha reta em km",
    "suficiencia_espacial": "matrizes produzidas apenas quando todas as coordenadas existentes estao auditadas",
    "proximidade_fora_municipio_aceita_m": 250
}, ensure_ascii=False, indent=2))
print("Instâncias exportadas para", saida / "instancias")

## 5. Auditoria a realizar antes de interpretação prática

- Identificar escolas com geocodificação aproximada, múltiplos prédios ou unidades anexas.
- Avaliar candidatos sobre zoneamento, terrenos disponíveis, restrições topográficas, acessibilidade e transporte.
- Validar \(s\) por etapa/turno junto à SEE-MG e à Secretaria Municipal de Educação, antes de qualquer recomendação pública.
- Construir distâncias viárias em análise futura; o experimento atual usa somente distâncias geométricas.
- Recalibrar participação pública com dados demográficos e administrativos adequados, não com a capacidade das unidades.